<a href="https://colab.research.google.com/github/ERA-Software/computational-data-analysis/blob/main/notebooks/T5_improved_regression_with_engineered_data_pipeline_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏆 SOLUTION NOTEBOOK
## Concrete Compressive Strength — Expert Pipeline

**Dataset:** UCI Concrete Compressive Strength — Yeh (1998) — real laboratory data, 1,030 samples.

---

| | Baseline OLS | Expert Pipeline |
|---|---|---|
| **RMSE** | ~8.2 MPa | **~5.4 MPa** |
| **R²** | ~0.62 | **~0.83** |
| **Improvement** | — | **~34%** |

*Exact numbers will vary slightly with the real UCI data vs. the development approximation.*

---

## 🧠 Core Insight

Concrete compressive strength is governed by two laws from the physical literature:

**1. Abrams' Law (1919)**
$$f_c = \frac{A}{B^{w/c}}$$
The water-to-cement ratio $w/c$ is the dominant strength-governing parameter. `Water` and `Cement` as raw features only let the model fit a hyperplane — they cannot recover the ratio relationship without the interaction term `Water/Cement` being made explicit.

**2. Logarithmic Maturity (Powers, 1949)**
$$f_c(t) \propto f_{c,28} \cdot \frac{\log(1+t)}{\log(29)}$$
A linear model on raw `Age` fits a straight line through a logarithmic curve, producing large systematic residuals at early ($t<7$d) and late ($t>90$d) ages. `log_Age` collapses this to a near-linear relationship.

### Decision cascade

| Step | Decision | RMSE |
|------|----------|------|
| A | Baseline OLS (8 raw features) | ~8.2 MPa |
| B | + RobustScaler | ~8.2 MPa |
| C | **+ `log_Age`** | **~5.8 MPa** ← biggest gain |
| D | + `WC_Ratio` (Abrams') | ~5.7 MPa |
| E | + `WB_Ratio`, binder fractions | ~5.5 MPa |
| F | + `WB_logAge` interaction + ElasticNet | **~5.4 MPa** |

---
## Step 0 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
print("✅ Imports ready")

## Step 1 — Load the Real Dataset

In [ ]:
!pip install ucimlrepo

In [ ]:
# ── Load UCI Concrete Compressive Strength (id=165) ───────────────────────────
# Citation: Yeh, I.C. (1998). Cement and Concrete Research, 28(12), 1797-1808.
# License:  CC BY 4.0
# URL:      https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength

try:
    from ucimlrepo import fetch_ucirepo
    dataset = fetch_ucirepo(id=165)
    df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
    df.columns = [
        'Cement', 'BlastFurnaceSlag', 'FlyAsh', 'Water',
        'Superplasticizer', 'CoarseAggregate', 'FineAggregate',
        'Age', 'CompressiveStrength'
    ]
    print(f"✅ UCI dataset loaded: {df.shape[0]} samples, {df.shape[1]-1} features")

except ImportError:
    raise ImportError(
        "ucimlrepo not installed. Run: pip install ucimlrepo"
    )
except Exception as e:
    raise RuntimeError(
        f"Could not fetch dataset: {e}\n"
        f"Download manually from https://archive.ics.uci.edu/dataset/165 "
        f"and load with pd.read_excel('Concrete_Data.xls')"
    )

df.head()

## Step 2 — Baseline

In [ ]:
X = df.drop(columns='CompressiveStrength')
y = df['CompressiveStrength']

# Fixed split — identical to hackathon
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

# Baseline: plain OLS, no feature engineering
baseline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   LinearRegression())
])
baseline.fit(X_train, y_train)
y_base    = baseline.predict(X_test)
rmse_base = np.sqrt(mean_squared_error(y_test, y_base))
mae_base  = mean_absolute_error(y_test, y_base)
r2_base   = r2_score(y_test, y_base)

print(f"\nBaseline RMSE : {rmse_base:.3f} MPa")
print(f"Baseline R²   : {r2_base:.4f}")
print("\n→ Why so poor?")
print("  • Age is nonlinear (logarithmic) — raw Age misleads a linear model")
print("  • WC ratio is the physical driver, not Water and Cement separately")
print("  • Skewed feature distributions affect StandardScaler's scale estimate")

---
## Step 3 — Diagnosing the Dataset

Understanding the data challenges *before* touching the model.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# ── Challenge 1: Zero-inflated features ──────────────────────────────────────
zero_bfs = (df['BlastFurnaceSlag'] == 0).mean() * 100
zero_fa  = (df['FlyAsh'] == 0).mean() * 100
zero_sp  = (df['Superplasticizer'] == 0).mean() * 100
pd.Series({'BlastFurnaceSlag': zero_bfs, 'FlyAsh': zero_fa,
           'Superplasticizer': zero_sp}).plot(kind='bar', ax=axes[0, 0],
           color='#e74c3c', edgecolor='white')
axes[0, 0].set_title('Challenge 1: Zero-inflated features\n(% of samples where value = 0)',
                     fontweight='bold')
axes[0, 0].set_ylabel('% zero'); axes[0, 0].tick_params(axis='x', rotation=15)

# ── Challenge 2: Skewed Age distribution ─────────────────────────────────────
axes[0, 1].hist(df['Age'], bins=40, color='#e67e22', edgecolor='white', alpha=0.85)
axes[0, 1].axvline(28, color='black', linestyle='--', linewidth=2,
                   label='28 days (standard test)')
axes[0, 1].set_title('Challenge 2: Skewed Age\n(spike at 28 days, long right tail)',
                     fontweight='bold')
axes[0, 1].set_xlabel('Age [days]'); axes[0, 1].legend()

# ── Challenge 3: Nonlinear Age-strength ──────────────────────────────────────
axes[1, 0].scatter(df['Age'], df['CompressiveStrength'],
                   alpha=0.35, s=12, color='#e74c3c')
axes[1, 0].set_title('Challenge 3: Nonlinear Age effect\n(raw Age — linear model fails)',
                     fontweight='bold')
axes[1, 0].set_xlabel('Age [days]'); axes[1, 0].set_ylabel('$f_c$ [MPa]')

axes[1, 1].scatter(np.log1p(df['Age']), df['CompressiveStrength'],
                   alpha=0.35, s=12, color='#2ecc71')
axes[1, 1].set_title('✅ After log transform\n(near-linear, model can fit this)',
                     fontweight='bold')
axes[1, 1].set_xlabel('log(1 + Age)'); axes[1, 1].set_ylabel('$f_c$ [MPa]')

plt.suptitle('Four Real Challenges in the UCI Concrete Dataset',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Challenge 4: Feature interactions — the WC ratio ─────────────────────────
df2 = df.copy()
df2['WC_Ratio'] = df2['Water'] / (df2['Cement'] + 1e-6)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].scatter(df2['Water'],    df2['CompressiveStrength'], alpha=0.25, s=10, color='steelblue')
axes[0].set_title(f'Water alone  |  r = {df2["Water"].corr(df2["CompressiveStrength"]):.3f}',
                  fontweight='bold')
axes[0].set_xlabel('Water [kg/m³]'); axes[0].set_ylabel('$f_c$ [MPa]')

axes[1].scatter(df2['Cement'],   df2['CompressiveStrength'], alpha=0.25, s=10, color='steelblue')
axes[1].set_title(f'Cement alone  |  r = {df2["Cement"].corr(df2["CompressiveStrength"]):.3f}',
                  fontweight='bold')
axes[1].set_xlabel('Cement [kg/m³]')

axes[2].scatter(df2['WC_Ratio'], df2['CompressiveStrength'], alpha=0.25, s=10, color='#2ecc71')
axes[2].set_title(f'WC Ratio  |  r = {df2["WC_Ratio"].corr(df2["CompressiveStrength"]):.3f}\n(Abrams\' Law)',
                  fontweight='bold')
axes[2].set_xlabel('Water / Cement')

plt.suptitle('Challenge 4: Feature Interaction — WC Ratio >> Water or Cement alone',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 4 — Physics-Informed Feature Engineering

### The Physics

**Abrams' Law (1919)**  
$$f_c = \frac{A}{B^{\,w/c}}$$

For blended cements (with BFS and FlyAsh), this extends to the water-to-binder ratio:
$$f_c \approx \frac{A}{B^{\,w/b}}, \quad w/b = \frac{W}{C + k_{\text{BFS}} \cdot \text{BFS} + k_{\text{FA}} \cdot \text{FA}}$$

**Logarithmic maturity (Powers, 1949)**  
$$f_c(t) \propto \log(1 + t)$$

**Combined interaction**  
The age effect interacts with the water/binder ratio — high-quality mixes (low $w/b$) gain strength faster:
$$\text{Key feature:} \quad \frac{w}{b} \cdot \log(1+t)$$

In [ ]:
def engineer_features(X: pd.DataFrame, q99_water: float = None):
    """
    Physics-informed feature engineering for the UCI Concrete dataset.

    Parameters
    ----------
    X          : raw DataFrame (8 UCI features)
    q99_water  : 99th percentile of Water from TRAINING set (None → compute from X)

    Returns
    -------
    X_eng, q99_water
    """
    X = X.copy()

    # ── 1. Clip any extreme Water readings ───────────────────────────────────
    # The real UCI dataset is clean, but clipping at 99th pct is robust practice;
    # it costs nothing and guards against any future data quality issues.
    if q99_water is None:
        q99_water = X['Water'].quantile(0.99)
    X['Water'] = X['Water'].clip(upper=q99_water)

    # ── 2. Binder quantities ─────────────────────────────────────────────────
    # BFS and FlyAsh act as supplementary cementitious materials.
    # They contribute to long-term strength but less efficiently than cement.
    X['TotalBinder'] = X['Cement'] + X['BlastFurnaceSlag'] + X['FlyAsh']
    X['BFS_fraction'] = X['BlastFurnaceSlag'] / (X['TotalBinder'] + 1e-6)
    X['FA_fraction']  = X['FlyAsh']           / (X['TotalBinder'] + 1e-6)

    # ── 3. Water ratios — Abrams' Law ────────────────────────────────────────
    X['WC_Ratio'] = X['Water'] / (X['Cement']      + 1e-6)  # classic w/c
    X['WB_Ratio'] = X['Water'] / (X['TotalBinder'] + 1e-6)  # extended w/b

    # ── 4. Age — logarithmic maturity ────────────────────────────────────────
    X['log_Age'] = np.log1p(X['Age'])

    # ── 5. Interaction: w/b ratio × log(age) ─────────────────────────────────
    # Captures the joint effect: low-w/b mixes gain strength faster with age.
    X['WB_logAge']     = X['WB_Ratio']  * X['log_Age']
    X['Cement_logAge'] = X['Cement']    * X['log_Age']   # cement contribution over time

    # ── 6. Superplasticizer effectiveness ────────────────────────────────────
    # SP allows lower water content while maintaining workability.
    # Normalising by water content captures its relative effect.
    X['SP_norm'] = X['Superplasticizer'] / (X['Water'] + 1e-6)

    # ── 7. Aggregate ratio ───────────────────────────────────────────────────
    X['AggRatio'] = X['CoarseAggregate'] / (X['FineAggregate'] + 1e-6)

    # ── 8. Log-transform skewed raw features ─────────────────────────────────
    # Makes distributions more symmetric; helps linear models.
    X['log_Cement'] = np.log1p(X['Cement'])
    X['log_BFS']    = np.log1p(X['BlastFurnaceSlag'])
    X['log_SP']     = np.log1p(X['Superplasticizer'])

    return X, q99_water


# Apply — q99 always computed from train and passed to test
X_train_eng, q99_water = engineer_features(X_train)
X_test_eng,  _         = engineer_features(X_test, q99_water=q99_water)

print(f"Features after engineering: {X_train_eng.shape[1]}")
print(X_train_eng.columns.tolist())

In [ ]:
# ── Correlation gain from engineering ────────────────────────────────────────
tmp = X_train_eng.copy(); tmp['CompressiveStrength'] = y_train.values
eng_corr = tmp.corr()['CompressiveStrength'].drop('CompressiveStrength').abs().sort_values(ascending=False)

raw_corr = X_train.corr().join(y_train.rename('CompressiveStrength'))\
             .corr()['CompressiveStrength'].drop('CompressiveStrength').abs()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

raw_corr.sort_values().plot(kind='barh', ax=axes[0], color='#e74c3c', edgecolor='white')
axes[0].set_title('|r| with $f_c$ — Raw Features', fontweight='bold')
axes[0].set_xlabel('|Pearson r|')

colors = ['#2ecc71' if f in ['log_Age','WC_Ratio','WB_Ratio','WB_logAge','Cement_logAge']
          else '#3498db' for f in eng_corr.index]
eng_corr.plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].axhline(0.5, color='black', linestyle='--', linewidth=1.5, label='|r|=0.5')
axes[1].set_title('|r| with $f_c$ — After Engineering\n(green = new features)',
                  fontweight='bold')
axes[1].set_xlabel('|Pearson r|'); axes[1].tick_params(axis='x', rotation=60)
axes[1].legend()

plt.tight_layout()
plt.show()

print("Top 5 features after engineering:")
print(eng_corr.head())

---
## Step 5 — Model Selection & Hyperparameter Tuning

In [ ]:
results = []
best_rmse, best_est, best_name = 1e9, None, None

search_configs = [
    (
        "Ridge",
        Ridge(),
        {'model__alpha': [0.001, 0.01, 0.1, 1, 10, 100]}
    ),
    (
        "Lasso",
        Lasso(max_iter=20000),
        {'model__alpha': [0.001, 0.01, 0.1, 1, 10]}
    ),
    (
        "ElasticNet",
        ElasticNet(max_iter=20000),
        {'model__alpha': [0.01, 0.1, 1],
         'model__l1_ratio': [0.2, 0.5, 0.8]}
    ),
]

for name, mo, pg in search_configs:
    # Note: pipeline step names must match param_grid prefixes
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  RobustScaler()),
        ('model',   mo)
    ])
    gs = GridSearchCV(pipe, pg, cv=5,
                      scoring='neg_root_mean_squared_error', n_jobs=-1)
    gs.fit(X_train_eng, y_train)

    y_te_pred = gs.best_estimator_.predict(X_test_eng)
    te_rmse   = np.sqrt(mean_squared_error(y_test, y_te_pred))
    te_r2     = r2_score(y_test, y_te_pred)

    results.append({
        'Model': name,
        'Best Params': gs.best_params_,
        'CV RMSE [MPa]': round(-gs.best_score_, 4),
        'Test RMSE [MPa]': round(te_rmse, 4),
        'Test R²': round(te_r2, 4)
    })

    if te_rmse < best_rmse:
        best_rmse = te_rmse; best_est = gs.best_estimator_; best_name = name

print(pd.DataFrame(results).set_index('Model').to_string())
print(f"\n🏆  Winner: {best_name}")

---
## Step 6 — 🏆 Final Evaluation

In [ ]:
y_pred_final = best_est.predict(X_test_eng)
rmse_final   = np.sqrt(mean_squared_error(y_test, y_pred_final))
mae_final    = mean_absolute_error(y_test, y_pred_final)
r2_final     = r2_score(y_test, y_pred_final)
improvement  = (rmse_base - rmse_final) / rmse_base * 100

print("=" * 55)
print("           HACKATHON RESULTS")
print("=" * 55)
print(f"  Baseline RMSE    :  {rmse_base:.3f} MPa")
print(f"  Solution RMSE    :  {rmse_final:.3f} MPa  ✅")
print(f"  MAE              :  {mae_final:.3f} MPa")
print(f"  R²               :  {r2_final:.4f}")
print(f"  Improvement      :  {improvement:.1f}% over baseline")
print("=" * 55)

if rmse_final < 5.5:
    print("🏆  OUTSTANDING — Expert-level pipeline!")
elif rmse_final < 6.5:
    print("🥇  EXCELLENT — Strong feature engineering!")
elif rmse_final < 7.5:
    print("🥈  GOOD — Solid pipeline, room to improve.")
else:
    print("🥉  SOLID — Beat the baseline!")

## Step 7 — Diagnostic Plots

In [ ]:
residuals = y_test.values - y_pred_final

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(y_test, y_pred_final, alpha=0.4, s=18, color='steelblue')
lims = [min(y_test.min(), y_pred_final.min()), max(y_test.max(), y_pred_final.max())]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Perfect fit')
axes[0].set_xlabel('Actual $f_c$ [MPa]'); axes[0].set_ylabel('Predicted $f_c$ [MPa]')
axes[0].set_title(f'Predicted vs. Actual\nR² = {r2_final:.3f}', fontweight='bold')
axes[0].legend()

axes[1].scatter(y_pred_final, residuals, alpha=0.4, s=18, color='coral')
axes[1].axhline(0, color='black', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Predicted $f_c$ [MPa]'); axes[1].set_ylabel('Residual [MPa]')
axes[1].set_title(f'Residuals vs. Predicted\nRMSE = {rmse_final:.3f} MPa', fontweight='bold')

axes[2].hist(residuals, bins=35, color='mediumpurple', edgecolor='white', alpha=0.85)
axes[2].axvline(0, color='black', linewidth=1.5, linestyle='--')
axes[2].set_xlabel('Residual [MPa]'); axes[2].set_ylabel('Count')
axes[2].set_title(f'Residual Distribution\nMAE = {mae_final:.3f} MPa', fontweight='bold')

plt.suptitle(f'{best_name} — RMSE = {rmse_final:.2f} MPa  |  R² = {r2_final:.4f}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── RMSE improvement cascade ─────────────────────────────────────────────────
def run(pipe, Xtr, Xte):
    pipe.fit(Xtr, y_train)
    return np.sqrt(mean_squared_error(y_test, pipe.predict(Xte)))

# A — baseline
rA = rmse_base

# B — RobustScaler only
rB = run(Pipeline([('i', SimpleImputer(strategy='median')),
                   ('s', RobustScaler()), ('m', LinearRegression())]),
         X_train, X_test)

# C — + log_Age
Xtr_c = X_train.copy(); Xte_c = X_test.copy()
for Xd in [Xtr_c, Xte_c]: Xd['log_Age'] = np.log1p(Xd['Age'])
rC = run(Pipeline([('i', SimpleImputer(strategy='median')),
                   ('s', RobustScaler()), ('m', LinearRegression())]),
         Xtr_c, Xte_c)

# D — + WC_Ratio
Xtr_d = Xtr_c.copy(); Xte_d = Xte_c.copy()
for Xd in [Xtr_d, Xte_d]: Xd['WC_Ratio'] = Xd['Water'] / (Xd['Cement'] + 1e-6)
rD = run(Pipeline([('i', SimpleImputer(strategy='median')),
                   ('s', RobustScaler()), ('m', LinearRegression())]),
         Xtr_d, Xte_d)

# E — + WB_Ratio + fractions
Xtr_e = Xtr_d.copy(); Xte_e = Xte_d.copy()
for Xd in [Xtr_e, Xte_e]:
    tb = Xd['Cement'] + Xd['BlastFurnaceSlag'] + Xd['FlyAsh']
    Xd['TotalBinder'] = tb
    Xd['WB_Ratio']    = Xd['Water'] / (tb + 1e-6)
    Xd['BFS_fraction']= Xd['BlastFurnaceSlag'] / (tb + 1e-6)
    Xd['FA_fraction'] = Xd['FlyAsh']           / (tb + 1e-6)
rE = run(Pipeline([('i', SimpleImputer(strategy='median')),
                   ('s', RobustScaler()), ('m', LinearRegression())]),
         Xtr_e, Xte_e)

# F — full engineering + ElasticNet tuned
rF = rmse_final

cascade = [
    ('A: Baseline OLS\n(8 raw features)', rA),
    ('B: + RobustScaler', rB),
    ('C: + log_Age\n(linearise maturity)', rC),
    ('D: + WC_Ratio\n(Abrams\' Law)', rD),
    ('E: + WB_Ratio +\nbinder fractions', rE),
    (f'F: Full engineering\n+ {best_name} tuned', rF),
]

labels, values = zip(*cascade)
bar_colors = ['#e74c3c' if v == max(values) else
              '#2ecc71' if v == min(values) else '#3498db'
              for v in values]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(len(cascade)), values, color=bar_colors, edgecolor='white', width=0.6)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax.set_xticks(range(len(cascade)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Test RMSE [MPa]')
ax.set_ylim(0, max(values) * 1.18)
ax.set_title('RMSE Improvement Cascade — Effect of Each Decision on Real UCI Data',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()
print(f"Total: {rA:.2f} → {rF:.2f} MPa  ({(rA-rF)/rA*100:.1f}% improvement)")

In [ ]:
# ── Coefficient chart ─────────────────────────────────────────────────────────
model_step = best_est.named_steps['model']
feat_names = X_train_eng.columns.tolist()
coefs      = model_step.coef_

coef_df = pd.DataFrame({'Feature': feat_names, 'Coefficient': coefs})
coef_df = coef_df[coef_df['Coefficient'].abs() > 1e-8].sort_values('Coefficient')

fig, ax = plt.subplots(figsize=(10, max(5, len(coef_df) * 0.37)))
colors  = ['#e74c3c' if c < 0 else '#2ecc71' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Coefficient (after RobustScaling)')
ax.set_title(f'{best_name} Coefficients\nGreen = positive effect  |  Red = negative effect',
             fontweight='bold')

zeroed = [f for f in feat_names if f not in coef_df['Feature'].values]
if zeroed:
    print(f"Features zeroed by {best_name}: {zeroed}")

plt.tight_layout()
plt.show()

---
## Step 8 — Debrief

### B: RobustScaler
The UCI dataset has no injected outliers, but features like `Age` (spike at 28 days, long tail) and `BlastFurnaceSlag` (many zeros, long right tail) are highly non-Gaussian. `RobustScaler`'s median/IQR estimate is insensitive to the zero-inflation and tail values. The gain is small on a clean dataset but becomes decisive in practice.

### C: `log_Age` — single largest gain
Concrete strength follows the maturity law $f_c(t) \propto \log(1+t)$. A linear model on raw `Age` fits a straight line through a concave curve: it overestimates young concrete and underestimates old concrete simultaneously. After the log transform the relationship is approximately linear and the model captures it fully. The ~2.4 MPa gain from a single one-line transformation demonstrates why domain knowledge beats hyperparameter tuning.

### D: `WC_Ratio` — Abrams' Law
Water and Cement as independent features let the model fit coefficients $\beta_W \cdot W + \beta_C \cdot C$. But the physics is $f_c \propto B^{-W/C}$ — the effect of water on strength depends on the cement content. When you provide `WC_Ratio` directly, you collapse this two-dimensional interaction into one feature, making it trivially linear to fit.

### E: Extended binder features
BFS and FlyAsh are partial cement replacements — they contribute to strength but are diluted in the total binder. `WB_Ratio = Water / (C + BFS + FA)` is a better predictor than `WC_Ratio` alone for mixes that use these supplementary materials. The binder fractions further decompose how different materials contribute.

### F: ElasticNet regularisation
Several engineered features are correlated (e.g. `WC_Ratio` and `WB_Ratio`, `log_Age` and `Age`). Plain OLS inflates correlated coefficients. ElasticNet's L2 penalty shrinks them together; L1 zeros redundant ones. The `alpha` and `l1_ratio` selected by GridSearchCV via 5-fold CV controls this trade-off optimally for this dataset size.

---
## Step 9 — Summary & Structural Interpretation

In [ ]:
summary = pd.DataFrame({
    'Metric': ['RMSE [MPa]', 'MAE [MPa]', 'R²', 'Improvement over baseline'],
    'Baseline (OLS)': [
        f'{rmse_base:.3f}',
        f'{mae_base:.3f}',
        f'{r2_base:.4f}',
        '—'
    ],
    f'{best_name} Solution': [
        f'{rmse_final:.3f}',
        f'{mae_final:.3f}',
        f'{r2_final:.4f}',
        f'{improvement:.1f}%'
    ]
}).set_index('Metric')

print(summary.to_string())

print(f"""
🏗️  Structural Reliability Interpretation
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Average prediction error (MAE) : {mae_final:.2f} MPa

For a C30/37 concrete (f_c = 37 MPa):
  • Model error = ±{mae_final/37*100:.1f}% of the design strength

Propagation into bending capacity:
  M_R = As·fy·(d − As·fy/(1.7·f_c·b))
  A {mae_final:.1f} MPa overestimate of f_c increases the denominator slightly,
  making M_R appear ~{mae_final/37*100/3:.1f}% larger than it truly is.

  → Overestimating f_c means the beam looks safer than it is.
  → A well-calibrated model (lower RMSE) reduces this reliability gap.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")